# CNN vs TCN cross-backbone comparison (plan §12 "Cross-backbone comparison").
#
# Consumes only committed CSV/JSON artifacts from the RQ1 and RQ2 runs —
# no training, no TensorFlow. Answers the six qualitative questions:
#   1. Does finer tokenization still trade clean performance for robustness?
#   2. Is BPE-500 still a useful middle point?
#   3. Do instability features predict coarse-tokenization failure?
#   4. Does adaptive routing beat the best fixed tokenizer on the frontier?
#   5. Does the router choose similar representations for similar inputs?
#   6. Are the same failure modes present?
# Plus the matched two-panel frontier figure (Macro-F1 vs CPU cost).

# # Typo Robustness v2 — CNN vs TCN comparison

In [ ]:
import glob
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from common import (
    CORRUPTION_LEVELS,
    FIGURES_DIR,
    RESULTS_DIR,
    ROUTER_EXPERTS,
    TRAINING_SEEDS,
    save_figure,
    save_json,
)

sns.set_theme(style="whitegrid")

# ## 1. Locate the latest RQ1 and RQ2 runs

In [ ]:
def latest_run(backbone: str) -> str:
    manifests = sorted(glob.glob(os.path.join(RESULTS_DIR, f"run_manifest_{backbone}_*.json")))
    if not manifests:
        raise FileNotFoundError(f"No run manifest for backbone '{backbone}' in {RESULTS_DIR}.")
    with open(manifests[-1], encoding="utf-8") as f:
        return json.load(f)["run_id"]


CNN_RUN_ID = latest_run("cnn")
TCN_RUN_ID = latest_run("tcn")
print(f"CNN run: {CNN_RUN_ID}\nTCN run: {TCN_RUN_ID}")

In [ ]:
def load_results(backbone: str, run_id: str) -> dict[str, pd.DataFrame]:
    paths = {
        "metrics": os.path.join(RESULTS_DIR, f"{backbone}_metrics_{run_id}.csv"),
        "predictions": os.path.join(RESULTS_DIR, f"{backbone}_predictions_{run_id}.csv"),
        "features": os.path.join(RESULTS_DIR, f"{backbone}_instability_features_{run_id}.csv"),
        "frontier": os.path.join(RESULTS_DIR, f"{backbone}_routing_frontier_{run_id}.csv"),
        "frontier_ood": os.path.join(RESULTS_DIR, f"{backbone}_routing_frontier_ood_{run_id}.csv"),
        "regret": os.path.join(RESULTS_DIR, f"{backbone}_routing_regret_{run_id}.csv"),
        "bpe_dropout": os.path.join(RESULTS_DIR, f"{backbone}_bpe_dropout_metrics_{run_id}.csv"),
        "latency": os.path.join(RESULTS_DIR, f"{backbone}_latency_{run_id}.csv"),
        "failure_modes": os.path.join(RESULTS_DIR, f"{backbone}_failure_mode_counts_{run_id}.csv"),
    }
    return {name: pd.read_csv(path) for name, path in paths.items() if os.path.exists(path)}


cnn = load_results("cnn", CNN_RUN_ID)
tcn = load_results("tcn", TCN_RUN_ID)

# ## 2. Q1 — Does finer tokenization trade clean performance for robustness?

In [ ]:
def clean_vs_degradation(metrics: pd.DataFrame) -> pd.DataFrame:
    clean = metrics[metrics["corruption_level"].eq(0)].groupby("model")["f1_macro"].mean()
    corrupted = metrics[metrics["corruption_level"].gt(0)].groupby("model")["f1_macro"].mean()
    clean_f1 = metrics[metrics["corruption_level"].eq(0)].groupby("model")["f1_macro"].first()
    rel_drop = (clean_f1 - corrupted) / clean_f1
    return pd.DataFrame({"clean_f1": clean_f1, "mean_corrupted_f1": corrupted, "relative_drop": rel_drop})


q1_table = pd.concat(
    {"cnn": clean_vs_degradation(cnn["metrics"]), "tcn": clean_vs_degradation(tcn["metrics"])}
)
display(q1_table.round(4))

# Spearman correlation between granularity rank and relative drop, per backbone.
from scipy.stats import spearmanr

granularity_rank = {"word": 0, "bpe_500": 1, "bpe_1000": 2, "bpe_2000": 3, "bpe_5000": 4, "char": 5}
q1_correlations = {}
for backbone, data in (("cnn", cnn), ("tcn", tcn)):
    table = clean_vs_degradation(data["metrics"])
    ranks = [granularity_rank[m] for m in table.index]
    q1_correlations[backbone] = {
        "spearman_rank_vs_relative_drop": float(spearmanr(ranks, table["relative_drop"]).statistic),
        "spearman_rank_vs_clean_f1": float(spearmanr(ranks, table["clean_f1"]).statistic),
    }
display(pd.DataFrame(q1_correlations).T)

# ## 3. Q2 — Is BPE-500 still a useful middle point?

In [ ]:
q2_rows = []
for backbone, data in (("cnn", cnn), ("tcn", tcn)):
    table = clean_vs_degradation(data["metrics"])
    for model in ROUTER_EXPERTS:
        row = table.loc[model]
        q2_rows.append(
            {
                "backbone": backbone,
                "model": model,
                "clean_f1": row["clean_f1"],
                "relative_drop": row["relative_drop"],
            }
        )
q2_df = pd.DataFrame(q2_rows)
display(q2_df.round(4))

# ## 4. Q3 — Do instability features predict failure under both backbones?

In [ ]:
q3_rows = []
for backbone, data in (("cnn", cnn), ("tcn", tcn)):
    if "features" not in data or "predictions" not in data:
        continue
    features = data["features"]
    predictions = data["predictions"]
    for expert in ("word", "char"):
        sub = predictions[predictions["model"].eq(expert)]
        correct_map = sub.set_index(["sample_id", "corruption_level"])["correct"]
        keys = list(features[["sample_id", "corruption_level"]].itertuples(index=False, name=None))
        y = np.asarray([correct_map.get(k, np.nan) for k in keys], dtype=float)
        valid = ~np.isnan(y)
        for feature in ("bpe_tokens_per_word", "fraction_words_split_2plus", "word_oov_fraction"):
            r = float(np.corrcoef(features.loc[valid, feature], y[valid])[0, 1])
            q3_rows.append({"backbone": backbone, "expert": expert, "feature": feature, "corr_with_correct": r})
q3_df = pd.DataFrame(q3_rows)
display(q3_df.round(4))

# ## 5. Q4 — Does adaptive routing beat the best fixed tokenizer?

In [ ]:
q4_rows = []
for backbone, data in (("cnn", cnn), ("tcn", tcn)):
    frontier = data["frontier"]
    fixed = frontier[frontier["system"].str.startswith("fixed_")]
    best_fixed = fixed.loc[fixed["macro_f1"].idxmax()]
    adaptive = frontier[frontier["system"].str.contains("learned")]
    for _, row in adaptive.iterrows():
        q4_rows.append(
            {
                "backbone": backbone,
                "system": row["system"],
                "macro_f1": row["macro_f1"],
                "best_fixed_f1": best_fixed["macro_f1"],
                "best_fixed_system": best_fixed["system"],
                "f1_delta": row["macro_f1"] - best_fixed["macro_f1"],
                "seq_len_delta_vs_best_fixed": row["mean_active_sequence_length"] - best_fixed["mean_active_sequence_length"],
            }
        )
q4_df = pd.DataFrame(q4_rows)
display(q4_df.round(4))

# ## 6. Q5 — Do routers make similar decisions for similar inputs?

In [ ]:
# Compare per-example chosen experts across backbones on shared keys.
routing_frames = []
for backbone, data in (("cnn", cnn), ("tcn", tcn)):
    if "regret" not in data:
        continue
    sub = data["regret"][["sample_id", "seed", "corruption_level", "chosen_expert"]].copy()
    sub["backbone"] = backbone
    routing_frames.append(sub)

if len(routing_frames) == 2:
    merged = routing_frames[0].merge(
        routing_frames[1],
        on=["sample_id", "seed", "corruption_level"],
        suffixes=("_cnn", "_tcn"),
    )
    agreement = (merged["chosen_expert_cnn"] == merged["chosen_expert_tcn"]).mean()
    confusion = pd.crosstab(merged["chosen_expert_cnn"], merged["chosen_expert_tcn"], normalize="all")
    print(f"Routing agreement across backbones: {agreement:.3f}")
    display(confusion.round(3))
    confusion.to_csv(os.path.join(RESULTS_DIR, f"routing_agreement_{CNN_RUN_ID}_{TCN_RUN_ID}.csv"))

# ## 7. Q6 — Are the same failure modes present?

In [ ]:
failure_frames = []
for backbone, data in (("cnn", cnn), ("tcn", tcn)):
    if "failure_modes" in data:
        counts = data["failure_modes"].rename(columns={"count": "n"})
        counts.insert(0, "backbone", backbone)
        failure_frames.append(counts)
if failure_frames:
    failure_comparison = pd.concat(failure_frames, ignore_index=True)
    display(failure_comparison)
    pivot = failure_comparison.pivot_table(index="failure_mode", columns="backbone", values="n", fill_value=0)
    display(pivot)

# ## 8. Matched two-panel frontier: Macro-F1 vs CPU inference cost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
for ax, (backbone, data) in zip(axes, (("cnn", cnn), ("tcn", tcn))):
    frontier = data["frontier"]
    latency = data.get("latency")
    if latency is not None:
        cost_map = latency.groupby("model")["latency_ms_per_sample"].mean().to_dict()
    else:
        cost_map = {}
    for i, system in enumerate(frontier["system"].unique()):
        sub = frontier[frontier["system"].eq(system)]
        is_curve = system in ("oracle", "fragmentation_threshold")
        ax.plot(
            sub["mean_active_sequence_length"],
            sub["macro_f1"],
            marker="o" if is_curve else "D",
            linestyle="-" if is_curve else "none",
            label=system,
            markersize=4,
        )
    ax.set_title(f"{backbone.upper()} backbone")
    ax.set_xlabel("Mean active sequence length (tokens)")
    ax.set_ylabel("Macro F1 (pooled over corruption levels)")
    ax.legend(fontsize=7)
fig.suptitle("Robustness-compute frontier — CNN vs TCN", fontsize=14)
save_figure(fig, "two_panel_frontier_cnn_vs_tcn", f"{CNN_RUN_ID}_{TCN_RUN_ID}")
plt.show()

# ## 9. Qualitative-conclusion summary (the deliverable of RQ2)

In [ ]:
summary_rows = []


def yes_no(condition: bool) -> str:
    return "YES" if condition else "NO"


for backbone, data in (("cnn", cnn), ("tcn", tcn)):
    table = clean_vs_degradation(data["metrics"])
    summary_rows.append(
        {
            "backbone": backbone,
            "Q1 finer=more robust": yes_no(table["relative_drop"].idxmin() == "char" or table.loc["char", "relative_drop"] < table.loc["word", "relative_drop"]),
            "Q2 BPE-500 middle": yes_no(
                table.loc["bpe_500", "clean_f1"] > table.loc["char", "clean_f1"]
                and table.loc["bpe_500", "relative_drop"] < table.loc["word", "relative_drop"]
            ),
            "Q4 adaptive beats best fixed": yes_no(
                (q4_df[q4_df["backbone"].eq(backbone)]["f1_delta"] > 0).any()
            ),
        }
    )
conclusions_df = pd.DataFrame(summary_rows)
display(conclusions_df)
conclusions_df.to_csv(
    os.path.join(RESULTS_DIR, f"cross_backbone_conclusions_{CNN_RUN_ID}_{TCN_RUN_ID}.csv"), index=False
)

In [ ]:
manifest = {
    "cnn_run_id": CNN_RUN_ID,
    "tcn_run_id": TCN_RUN_ID,
    "questions_answered": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6"],
}
save_json(
    manifest,
    os.path.join(RESULTS_DIR, f"run_manifest_comparison_{CNN_RUN_ID}_{TCN_RUN_ID}.json"),
)
print("Cross-backbone comparison complete.")